In [1]:
import pandas as pd
import os
import pyarrow.parquet as pq
from concurrent.futures import ProcessPoolExecutor
from plotly.subplots import make_subplots
import plotly.express as px

import plotly.graph_objects as go

In [19]:
filenames = [os.path.join(os.getcwd(), "peaks-tables", "27_qc_2x_dil_milliq.parquet")]

In [20]:
with ProcessPoolExecutor(max_workers=4) as executor:
    samples = executor.map(pd.read_parquet, filenames)

In [22]:
plots_per_row = 2
n_files = len(filenames)
n_rows = (n_files + plots_per_row - 1) // plots_per_row

# Pad titles so the list matches the total number of subplot slots.
subplot_titles = filenames

fig = make_subplots(
    rows=n_rows,
    cols=plots_per_row,
    subplot_titles=subplot_titles,
    shared_xaxes='all',
    shared_yaxes='all'
)

for i, peaks in enumerate(samples):
    row = i // plots_per_row + 1
    col = i % plots_per_row + 1

    fig.add_trace(
        go.Scatter(
            x=peaks['rt'],
            y=peaks['mz'],
            c=peaks['area'],
            size=peaks['sd1']+peaks['sd2']
        ),
        row=row,
        col=col
    )

In [ ]:
fig.show()